# Real-IAD 데이터 분석 + 증강·학습 과제 · scikit-image + SciPy

**이번 실습 범위**
1. 클래스별 밝기: 품목 × 불량 유형 × 카메라별로 비교
2. GAN 참고용 불량 위치 히트맵: train의 불량 마스크로 생성
3. 불량 마스크 → 연결 성분 → 바운딩박스 → CSV·COCO·YOLO 라벨
4. 모델 입력 전처리: **1024 / 512 / 256** RGB·마스크·작은 불량 보존 비교
5. **학생 과제:** 밝기 및 OpenCV 증강을 구현하고 실제 학습으로 효과 평가

1~4는 실행 가능한 분석 코드, 5는 단계·힌트·빈 함수·결과표만 제공합니다.
GAN 모델 구현·증강 정답·학습 정답은 제공하지 않습니다.
기본 실행은 **합성 데모**이며 실제 Real-IAD 통계나 모델 성능이 아닙니다.
학습 데이터 전체를 메모리에 올리지 않으며, 기본 분석에는 GPU·LLM·API 키가 필요 없습니다.
`analysis_guide_legacy.ipynb`는 이전 일반 분석 흐름을 보존한 참고본입니다.

## 0. 설치·경로·실험 조건

압축을 풀고 이 노트북과 core.py가 있는 폴더에서 실행합니다.
터미널에서 `python -m pip install -r requirements-notebook.txt` 후 `python -m jupyter lab`.
실제 데이터는 `USE_DEMO=False`, ROOT·JSON_DIR·CATEGORIES를 변경하세요.
OpenCV 증강 과제는 두 버전 모두 requirements-assignment.txt가 추가로 필요합니다.

**클래스 정의:** category는 제품 품목(toothbrush/toy), anomaly_class는 제품의 불량 코드(OK/HS/ZW 등)입니다.
불량 제품의 모든 시점에 결함이 보이는 것은 아닙니다. 밝기 그래프의 클래스는 제품 수준 라벨입니다.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Image as DisplayImage
if not Path('core.py').is_file():
    raise RuntimeError('노트북과 core.py가 있는 폴더에서 실행하세요.')
from core import build_manifest, audit, project_split, analyze, read_rgb, read_mask, nearest, gray
from coursework import (run_focused, mask_to_boxes, coarse_occupancy, show_boxes)
from backend import NAME
from make_demo import generate
get_ipython().run_line_magic('matplotlib', 'inline')

USE_DEMO = True
SEED = 42
SIZES = (1024, 512, 256)
EXPECTED_VIEWS = ('C1','C2','C3','C4','C5')
CATEGORIES = None  # 실제 데이터 예: ['toothbrush','toy']
MAX_PRODUCTS = 12  # 0이면 선택한 개발 분할의 전체 제품
SPLITS = ('train','val','validation')
USE_SUPERVISED_PROJECT_SPLIT = False
HOLDOUT_DEFECTS = ()
ROOT = Path('demo_notebook/images') if USE_DEMO else Path('/data/Real-IAD/realiad_1024')
JSON_DIR = Path('demo_notebook/jsons') if USE_DEMO else Path('/data/Real-IAD/realiad_jsons')
OUTPUT = Path('notebook_results_demo' if USE_DEMO else 'notebook_results_real_v2')
OUTPUT.mkdir(parents=True, exist_ok=True)
if USE_DEMO and not (JSON_DIR/'toothbrush.json').is_file():
    generate('demo_notebook', seed=2026)  # source 512 x 640; 1024 is upscaling
print(NAME, ROOT.resolve(), 'SYNTHETIC ONLY' if USE_DEMO else 'REAL DATA')
print('sizes:', SIZES, '| locked test excluded from EDA')

## 1. 매니페스트·제품 분할·무결성

같은 제품의 5장은 같은 분할에 둡니다. 밝기·GAN·해상도 설계에 잠근 test는 사용하지 않습니다.
공식 train에는 정상만 있을 수 있습니다. 이때 train 불량 히트맵이 비어 있는 것이 정상 동작입니다.
지도 탐지/GAN을 위해 불량을 train에 쓰려면 프로젝트 제품 분할을 명시적으로 활성화하고 공식 분할과 구분하세요.

아래 감사는 test를 포함한 전체 파일의 읽기·키·해시·라벨 정합성만 검사합니다.
개발 EDA는 선택한 train/val에서만 수행합니다. mask missing/error를 정상으로 대체하지 않습니다.
이진 마스크만 지원하며 다중 클래스·ignore 값은 검수된 명시적 매핑이 필요합니다.

In [ ]:
manifest = build_manifest(ROOT, JSON_DIR, OUTPUT/'input', CATEGORIES)
if USE_SUPERVISED_PROJECT_SPLIT:
    manifest = project_split(manifest, seed=SEED, holdout_defects=HOLDOUT_DEFECTS)
checked, issues = audit(ROOT, manifest, OUTPUT/'audit', EXPECTED_VIEWS)
display(issues)
print('products:', len(manifest[['category','sample_key']].drop_duplicates()))
display(manifest.groupby(['category','project_split','anomaly_class']).size().rename('n_images').reset_index())
assert manifest.groupby(['category','sample_key']).project_split.nunique().le(1).all()


## 2. 공통 개발 표본 확정 + 네 분석 실행

기존 analyze의 표본 선정과 감사를 유지합니다. focused 분석도 반드시 같은 selected_manifest를 사용합니다.
`MAX_PRODUCTS`는 제품 수이며 각 제품의 5뷰가 함께 선택됩니다.
대용량 실제 데이터에서는 전체 감사를 한 번 검토한 뒤 결과 경로와 분할 해시를 보존하세요.
기존 일반 분석 결과는 legacy_analysis에, 이번 결과는 focused_analysis에 저장합니다.

In [ ]:
metrics, product_summary = analyze(ROOT, manifest, OUTPUT/'legacy_analysis',
    splits=SPLITS, sizes=SIZES, expected=EXPECTED_VIEWS,
    max_products=MAX_PRODUCTS, seed=SEED)
selected = pd.read_csv(OUTPUT/'legacy_analysis/selected_manifest.csv', keep_default_na=False)
assert not selected.project_split.eq('test').any()
RESULTS = OUTPUT/'focused_analysis'
result = run_focused(ROOT, selected, RESULTS)
print('selected images:', len(selected), '| selected products:',len(product_summary))
display(selected[['category','sample_key','view_id','anomaly_class','project_split','mask_state']].head())


## 3. 클래스별 밝기 분석

**단계**: RGB 공통 디코딩 → gray=0.299R+0.587G+0.114B → 이미지별 mean·median·P05·P95 → 품목×불량 코드×뷰 집계.
한 영상의 픽셀을 모두 독립 표본으로 보지 말고 제품 수도 확인합니다.
히스토그램은 각 이미지를 합계 1로 정규화한 뒤 평균하므로 큰 이미지가 더 큰 가중치를 갖지 않습니다.

- 전체 영상의 밝기 차이는 배경·카메라·촬영 조건 영향도 포함합니다.
- mask 양성 내부(defect_mean)와 그 밖(nondefect_mean)도 비교합니다.
- 마스크 밖은 제품 정상 표면+배경이 섞인 영역이며 **순수 배경/정상 제품 정답**이 아닙니다.
- 밝기 차이가 있다는 이유만으로 그 값이 결함의 원인이라고 결론내리지 않습니다.

In [ ]:
brightness, brightness_summary, brightness_hist = result['brightness']
display(brightness_summary)
print('class columns: category=제품 품목, anomaly_class=불량 유형')
for path in sorted((RESULTS/'brightness').glob('*.png'))[:2]:
    display(DisplayImage(filename=str(path), width=1000))
# 나머지 그림은 focused_analysis/brightness/에서 확인하세요.


### 밝기 분석 해석 과제

아래 표의 빈칸을 직접 작성하세요. 품목과 view를 고정한 비교를 최소 2개 포함합니다.
평균만 보지 말고 분포 겹침·표본 수·배경 영향·결함 부위의 상대 밝기를 근거로 적으세요.
밝기 증강 범위는 이 분석과 시각 검수로 정하고 test 통계로 정하지 않습니다.

In [ ]:
brightness_findings = pd.DataFrame([
    {'category':'', 'view_id':'', 'class_pair':'', 'n_products':'',
     'observation':'', 'possible_confounder':'', 'augmentation_hypothesis':''}
    for _ in range(2)])
display(brightness_findings)


## 4. GAN 참고용 불량 위치 히트맵

산점도 대신 **정규화 위치 격자(기본 64×64)의 영상별 점유율**을 만듭니다.
품목 × 불량 유형 × 카메라를 분리하며 **project_split=train만** 사용합니다.
양성 원본 픽셀을 격자에 매핑하므로 1픽셀 불량도 누락하지 않습니다. 픽셀 밀도나 GAN 생성 결과가 아닙니다.

- `occupancy_known`: 해당 train 그룹의 확인 가능한 마스크 수(빈 마스크 포함)로 나눈 점유율
- `occupancy_positive`: 양성 마스크 영상 수만으로 나눈 조건부 점유율
- 미제공·오류는 분모에서 제외하고 n_unknown으로 표시
- train에 양성 마스크가 0개면 positive 맵은 NA, 데이터가 없으면 그룹 자체가 없을 수 있음

**GAN 활용 힌트**: 소스 제품·유형·뷰에 맞는 위치 후보를 제안하는 참고 자료입니다.
제품 전경·부위 검수 없이 확률맵만 따라 배치하지 마세요. 5뷰 좌표를 섞거나 동일 물리 결함으로 연결하지 않습니다.
선택 표본으로 만든 히트맵이므로 실제 GAN 준비에서는 train 전체로 다시 산출하세요.

In [ ]:
heatmap_info = result['heatmaps']
display(heatmap_info)
source = pd.read_csv(RESULTS/'gan_heatmaps/gan_source_manifest.csv', keep_default_na=False)
assert source.empty or source.project_split.eq('train').all()
if heatmap_info.empty:
    print('train 불량 그룹 없음: test를 자동으로 가져오지 않습니다.')
else:
    positive_groups = heatmap_info[heatmap_info.n_positive.gt(0)]
    for rec in positive_groups.head(2).itertuples():
        display(DisplayImage(filename=str(RESULTS/'gan_heatmaps'/f'{rec.heatmap_id}.png'), width=900))
        a = np.load(RESULTS/'gan_heatmaps'/rec.npz)
        print(rec.category, rec.anomaly_class, rec.view_id,
              'known=',rec.n_known,'positive=',rec.n_positive,'unknown=',rec.n_unknown)


### GAN 위치 설계 과제 (생성기는 구현하지 않음)

1. train의 품목·유형·뷰별 양성 제품 수를 확인합니다.
2. 히트맵과 원본 5뷰·실제 결함 마스크를 대조합니다.
3. 합성 가능한 전경/부위 ROI를 검수해서 별도 정의합니다.
4. 위치 후보·마스크 크기·방향·강도의 표본화 방법을 설명합니다.
5. 편집 마스크와 실제 생성 불량 마스크의 차이, 박스 승인 방법을 정합니다.
6. source_sample_key, view_id, seed, generator_version, approved_mask_ref를 기록할 양식을 설계합니다.

**제출:** GAN 위치 설계표 + 허용·거절 위치 예시. 지금 단계에서는 GAN 학습·생성 함수를 요구하지 않습니다.

In [ ]:
gan_plan = pd.DataFrame(columns=['category','anomaly_class','view_id','n_train_products',
    'allowed_roi','position_policy','mask_size_policy','review_rule'])
display(gan_plan)


## 5. 마스크 → 바운딩박스 추출

**단계**: 검증한 바이너리 마스크 → 8방향 연결 성분 → 각 성분의 최소 박스 → 원본 좌표 저장.
미세 결함 보존을 위해 기본 면적 제거·closing·팽창·박스 padding을 하지 않습니다.
분리된 성분은 각각 박스이며 하나의 물리적 결함이 여러 성분으로 나뉠 수도 있습니다.

좌표는 `[x0,y0,x1,y1]`, 오른쪽·아래 끝점 제외입니다.
예: 마스크가 x=100..119,y=200..209이면 `[100,200,120,210]`.
COCO는 `[x0,y0,w,h]`, YOLO는 `class cx/W cy/H w/W h/H`.
원본 불량 코드를 성분별 정답으로 추정하지 않고 **defect 1클래스**로 내보냅니다.

In [ ]:
boxes, detection_images = result['boxes']
display(boxes.head(15))
display(detection_images.head())
print('bbox CSV:',RESULTS/'boxes/bbox_annotations.csv')
print('excluded:',RESULTS/'boxes/bbox_excluded.csv')
if not boxes.empty:
    selected_id = int(boxes.iloc[0].image_id)
    image_row = detection_images[detection_images.image_id.eq(selected_id)].iloc[0]
    fig = show_boxes(ROOT,image_row,boxes)
    plt.show(); plt.close(fig)
    assert (boxes.x1 > boxes.x0).all() and (boxes.y1 > boxes.y0).all()
    assert (boxes.x1 <= boxes.width).all() and (boxes.y1 <= boxes.height).all()


### 내보내기 확인

- CSV: 원본 xyxy·정규화 xywh·제품/뷰 키·성분 면적·박스 면적
- COCO: split별 JSON, category_id=1=defect, images에는 확인된 빈 마스크 뷰도 포함
- YOLO: `yolo_labels/<split>/<image_id>.txt`, class_id=0=defect
- **이미지는 복사하지 않았습니다.** detection_images.csv의 image_path→yolo_stem 매핑에 따라
  학습용 이미지를 같은 stem으로 준비하는 단계는 과제입니다. 원본 이름에 바로 붙이면 짝이 안 맞습니다.
- 미제공·오류는 제외 사유를 저장하며 정상 빈 txt로 만들지 않습니다.

원본 GT 박스 생성과 실제 RGB 기반 탐지 추론은 서로 다른 단계입니다.

In [ ]:
label_paths = sorted((RESULTS/'boxes/yolo_labels').rglob('*.txt'))
if label_paths:
    print(label_paths[0], '\n', label_paths[0].read_text())
print((RESULTS/'boxes/YOLO_README.md').read_text())


## 6. 입력 해상도 1024 / 512 / 256 비교

**완성된 분석:** 동일 RGB의 bilinear resize, 성분 ID 맵의 명시적인 pixel-center nearest,
원본 성분별 잔존 픽셀·소실·축소 후 박스 폭·높이, 입력 메모리 산술 비교.
배열은 RGB uint8이며 모델 입력 `[5,3,H,W]` float32로 바뀔 때의 **입력 텐서만** 계산합니다.
활성값·가중치·작업공간·GPU 피크 메모리나 추론 속도는 포함하지 않습니다.

| 정사각형 입력 | 1뷰 FP32 입력 MiB | 5뷰 FP32 입력 MiB |
|---|---:|---:|
|1024|12|60|
|512|3|15|
|256|0.75|3.75|

합성 데모 원본은 512×640이므로 1024는 업샘플입니다. 원본에 없는 디테일을 복원하지 않습니다.
비정사각 원본의 square resize는 종횡비를 바꿉니다. 실제 학습에서 letterbox를 선택하면
박스·mask 변환과 역변환을 따로 구현하고 이번 분석과 다른 정책임을 기록하세요.
모델 내부에서 다시 800 등으로 resize하면 256/512/1024 비교가 무효가 되므로 입력 hook으로 확인합니다.

In [ ]:
res_images, res_components, res_summary = result['resolutions']
display(res_summary)
print('disappearance = 원본 성분이 최근접 축소 후 0픽셀인 비율; 모델 Recall이 아님')
# 원본·같은 crop 확대를 함께 보고, 축소 이미지의 작은 불량을 육안 확인합니다.
positive = selected[(selected.status=='ok') & (selected.mask_state=='positive')]
if not positive.empty:
    rec = positive.iloc[0]
    rgb = read_rgb(ROOT/rec.image_path)
    mask = read_mask(ROOT/rec.mask_path,rgb.shape[:2])
    from PIL import Image as PILImage
    fig,axes = plt.subplots(2,3,figsize=(12,7))
    for j,size in enumerate(SIZES):
        rr=np.asarray(PILImage.fromarray(rgb).resize((size,size),PILImage.Resampling.BILINEAR))
        mm=nearest(mask,size)
        axes[0,j].imshow(rr);axes[0,j].set_title(f'RGB {size}')
        axes[1,j].imshow(mm,cmap='gray',interpolation='nearest');axes[1,j].set_title(f'Mask pixels={mm.sum()}')
        axes[0,j].axis('off');axes[1,j].axis('off')
    plt.tight_layout();plt.show();plt.close(fig)


## 7. 학생 과제 안내: 밝기·OpenCV 증강 → 학습

**여기부터는 정답 코드가 없습니다.** 빈 함수를 구현하고 실행 스위치를 켜세요.
기본 Run All은 분석 부분과 빈 결과표까지 실행되며 과제 함수·학습·가중치 다운로드는 실행하지 않습니다.
GAN 구현은 이번 과제 범위 밖이며 히트맵 분석 결과만 향후 연결합니다.

### 과제 A. 밝기 증강

해야 하는 순서:
1. train의 클래스·뷰별 밝기 분포에서 증강 필요성과 범위를 제안
2. RGB uint8 → 넓은 자료형/float에서 연산 → clipping → uint8 반환
3. gamma 또는 alpha·beta 방식 중 두 후보를 정의하고 원본/약/중/강 격자 작성
4. 마스크와 박스는 밝기 변환에서 좌표를 유지
5. 과포화·암부 비율, 결함 대비, 실제 보임 여부로 허용 범위를 고정

힌트: uint8 배열에 단순 덧셈하면 overflow가 생길 수 있습니다.
`cv2.convertScaleAbs`는 절댓값 처리도 하므로 음수 beta 설계에서 의도와 맞는지 확인하세요.
밝기 증강은 `cv2.LUT`, `cv2.addWeighted`, float 연산 등을 검토할 수 있습니다.
**실습 데이터는 RGB**이며 cv2.imread의 기본 BGR와 혼동하지 마세요.
정상/불량에 공통 정책부터 적용하고 클래스별 정책은 별도 실험으로 기록하세요.

In [ ]:
def student_brightness(rgb, *, rng, config):
    """TODO: RGB uint8 HWC -> RGB uint8 HWC, same shape, deterministic with rng."""
    raise NotImplementedError('과제 A: 밝기 증강을 직접 구현하세요.')

RUN_BRIGHTNESS_ASSIGNMENT = False
if RUN_BRIGHTNESS_ASSIGNMENT:
    train_source = selected[selected.project_split.eq('train') & selected.status.eq('ok')]
    if train_source.empty:
        raise ValueError('증강 소스에 사용할 train 이미지가 없습니다.')
    candidate = student_brightness(read_rgb(ROOT/train_source.iloc[0].image_path),
        rng=np.random.default_rng(SEED),config={})
    # TODO: 원본/증강, 밝기 분포, 포화율·결함 대비와 동일 좌표 유지 검증


### 과제 B. OpenCV 기하·광학 증강과 라벨 동기화

1. 학습 카메라 의미를 유지하는 flip·작은 rotate·translate 후보를 고릅니다.
2. 기하 변환 행렬과 난수를 **한 번** 뽑고 RGB·mask에 동일하게 적용합니다.
3. RGB는 적절한 연속 보간, mask는 nearest, 경계 채움 정책을 명시합니다.
4. 변환된 mask에서 `mask_to_boxes`로 박스를 **다시 추출**합니다.
5. 잘려서 사라진 결함은 빈 라벨로 처리하되, 해당 합성 뷰 라벨·제품 묶음 관계도 검토합니다.
6. blur·noise·contrast 등 광학 증강은 불량 흔적이 지워지지 않는 강도를 검수합니다.

힌트 API: `cv2.flip`, `cv2.getRotationMatrix2D`, `cv2.warpAffine`, `cv2.GaussianBlur`, `cv2.LUT`.
직접 박스를 변환한다면 네 모서리를 변환한 뒤 범위를 제한하세요. 회전 시 두 모서리만 옮기면 틀릴 수 있습니다.
가짜 불량을 만든 광학 잡음과 기존 실제 결함 GT가 맞는지 검수하세요.
**다른 제품의 5뷰를 섞지 말고**, augmentation seed와 source_sample_key를 기록하세요.

In [ ]:
def student_opencv_augment(rgb, mask, *, rng, config):
    """TODO: return augmented_rgb, augmented_binary_mask, transform_metadata.
    광학 변환만이면 mask를 유지. 기하 변환이면 같은 변환을 적용.
    """
    raise NotImplementedError('과제 B: OpenCV 증강과 마스크 동기화를 구현하세요.')

RUN_OPENCV_ASSIGNMENT = False
if RUN_OPENCV_ASSIGNMENT:
    import cv2  # requirements-assignment.txt 설치 후 사용
    if positive.empty:
        raise ValueError('train 양성 마스크 제품부터 준비하세요.')
    train_positive=positive[positive.project_split.eq('train')]
    if train_positive.empty:
        raise ValueError('과제 증강 소스에 사용할 train 양성 마스크가 없습니다.')
    rec=train_positive.iloc[0]
    x=read_rgb(ROOT/rec.image_path);m=read_mask(ROOT/rec.mask_path,x.shape[:2])
    aug_x,aug_m,transform=student_opencv_augment(x,m,rng=np.random.default_rng(SEED),config={})
    aug_boxes=mask_to_boxes(aug_m)
    # TODO: dtype/shape/결함 보존/원본 분할 계보/박스 오버레이 검증 후 승인


### 과제 C. 1024/512/256 모델 입력 파이프라인

1. 같은 원본 제품 분할을 고정하고 각 크기의 resize/letterbox 방식을 선언합니다.
2. RGB·mask·박스를 동기 변환하고 변환 메타데이터를 남깁니다.
3. RGB HWC uint8 → 모델 요구 CHW float·정규화. 탐지기의 내장 전처리와 중복 적용하지 않습니다.
4. 배치 직전과 실제 백본 입력 직전 크기를 확인합니다.
5. 입력 크기·최대 epoch/step·유효 batch·사전학습 가중치·seed를 기록합니다.
6. 학습 시간·피크 메모리·제품 Recall·박스 AP50/AP@[.50:.95]·작은 불량 Recall을 비교합니다.

힌트: 메모리 제한으로 1024 batch가 줄어들면 accumulation과 유효 batch를 기록하세요.
작은 불량 mask 소실률을 모델 탐지 Recall로 대신 쓰면 안 됩니다.
검증/test는 고정된 결정적 전처리만 적용하고 **랜덤 증강은 train에만** 적용합니다.

In [ ]:
def student_prepare_input(rgb, mask, *, target_size, config):
    """TODO: model input + transformed mask + boxes + transform metadata.
    1024/512/256의 실제 네트워크 입력 크기까지 확인할 것.
    """
    raise NotImplementedError('과제 C: 모델 입력 전처리를 구현하세요.')


### 과제 D. 증강 후 실제 학습·평가

공개 사전학습 가중치가 있는 탐지기 **하나**를 먼저 선택해 데이터 파이프라인을 검증합니다.
모델 12종 전체 비교는 프로젝트 후속 단계이며 이번 과제에서 한꺼번에 수행하지 않습니다.

| 단계 | 해야 할 작업 | 통과 기준 |
|---|---|---|
|D1|COCO 또는 YOLO dataset 구성, 원본→라벨 ID 연결|빈 라벨·다중 박스·잘린 결함 확인|
|D2|사전학습 모델 로드·불량 클래스 헤드 설정|학습 가능한 파라미터·로드 제외 키 기록|
|D3|소규모 train 제품으로 과적합 확인|loss 감소·박스가 결함 위치를 덮음|
|D4|고정 split에서 baseline 학습|validation 선택, test 미사용|
|D5|동일 조건으로 밝기/OpenCV/결합 증강 비교|증강 수·노출 step·seed·유효 batch 기록|
|D6|상위 증강 정책의 1024/512/256 비교|실제 입력 크기·성능·비용 비교|
|D7|validation에서 정책 확정, test 최종 평가|반복 test 튜닝 금지|

**권장 실험 수:** 512에서 증강 4조건 + 선정 정책의 1024·256 추가 = 총 6조건.
여유가 있으면 3해상도×4증강=12조건으로 확대합니다.
내장 Mosaic·HSV 등은 baseline에서 끄거나 전 조건에 동일 적용해 추가 OpenCV 효과와 구분하세요.
제품 판정은 5뷰 관찰을 모아 평가하고 이미지 단위 지표와 구분합니다.

In [ ]:
AUGMENTATIONS = ('none','brightness','opencv','brightness_opencv')
training_plan = pd.DataFrame([
    {'experiment_id':f'e{i+1:02d}','input_size':512,'augmentation':policy,
     'seed':SEED,'split_hash':'TODO','model_id':'TODO','pretrained_id':'TODO',
     'effective_batch':'TODO','max_steps':'TODO','status':'NOT_RUN'}
    for i,policy in enumerate(AUGMENTATIONS)])
# 선정 정책을 정한 뒤 1024·256 행을 추가하세요.
training_results = pd.DataFrame(columns=['experiment_id','actual_input_size','n_train_products',
    'product_recall','product_fpr','box_ap50','box_map_50_95','small_defect_recall',
    'five_view_p95_ms','peak_memory_mib','train_seconds','status'])
display(training_plan);display(training_results)
training_plan.to_csv(OUTPUT/'training_plan_TEMPLATE.csv',index=False)
training_results.to_csv(OUTPUT/'training_results_EMPTY.csv',index=False)

def student_train_and_evaluate(experiment, *, train_manifest, validation_manifest):
    """TODO: model init, dataloader, train, checkpoint selection, held-out validation.
    Return measured results only; test is not passed to model selection.
    """
    raise NotImplementedError('과제 D: 학습·검증 루프와 지표를 직접 구현하세요.')

RUN_TRAINING_ASSIGNMENT = False
if RUN_TRAINING_ASSIGNMENT:
    raise NotImplementedError('먼저 train/val 데이터로더·학습·평가·내장증강 설정을 연결하세요.')


## 8. 제출물·평가 기준·실패 사례

- **필수 분석:** 클래스별 밝기 표·그림·해석, train-only 히트맵+분모+GAN 위치 제안,
  bbox CSV/오버레이/라벨 매핑, 해상도별 소실·박스 크기·RGB·입력 메모리 비교
- **과제:** 밝기 함수, OpenCV 동기 증강 함수, 모델 입력 변환 함수, 학습·검증 루프,
  train 전용 증강과 동일 원본 분할의 증거, 실제 측정 결과표, 실패 사례
- **점수 예시:** 분석 40점(밝기/히트맵/박스/해상도 각 10), 증강·라벨 정합성 25,
  학습·공정 비교 25, 재현·해석 10. 수업에 맞게 조정 가능
- **필수 실패 확인:** overflow, RGB/BGR 혼동, mask bilinear 보간, 미제공 mask를 정상화,
  test 누수, 다른 뷰 좌표 병합, 박스 원본 좌표 불일치, 모델 내부 강제 resize
- 결과 숫자를 채우기 위해 임의 점수·학습 성공·가짜 지연을 만들지 않습니다.
  실패·미실행은 그대로 기록하고 실제 데이터 학습과 합성 데모 분석을 구분하세요.

### 두 구현 교차 확인
`core.py`의 기존 비교기는 일반 분석용입니다. 이번 네 분석은 `compare_focused.py`로 별도 비교합니다.
두 버전에서 동일 selected_manifest·split·크기·grid 설정을 써야 합니다.
시간 값은 비교 대상에서 제외하며 수학 결과·라벨·heatmap 배열을 대조합니다.

In [ ]:
print('analysis output:', RESULTS.resolve())
print('현재 실행은 분석 데모입니다. 증강·학습 과제의 status는 NOT_RUN입니다.')
print('교차 비교 예: python compare_focused.py <OpenCV focused_analysis> <ScikitImage focused_analysis>')


## 참고 API

- OpenCV 성분·박스: https://docs.opencv.org/4.x/d3/dc0/group__imgproc__shape.html
- OpenCV dtype·픽셀 연산: https://docs.opencv.org/4.x/d3/df2/tutorial_py_basic_ops.html
- OpenCV 기하 변환·보간: https://docs.opencv.org/4.x/da/d54/group__imgproc__transform.html
- scikit-image label·regionprops: https://scikit-image.org/docs/stable/api/skimage.measure.html
- 실제 마스크 값·카메라 매핑은 내려받은 Real-IAD 릴리스에서 먼저 검증합니다.
